# Úloha č.1:

### Znenie

Naimportujte si potrebné knižnice.

### Riešenie

In [ ]:
import torch
import torch.nn as nn 
import torch.nn.functional as F 
import torchvision 
import torchvision.transforms as transforms
import matplotlib.pyplot as plt 
from tqdm import tqdm
from datetime import timedelta
import time

### Vysvetlenie

Naimportované boli balíky knižnice *pytorch*, slúžiace na tvorbu neurónových sietí. Takisto bola naimportovaná knižnica *torchvision* a jej balík *torchvision.transforms*, slúžiaca na prácu s obrázkovými dátami a ich transformácie. Ďalej importované knižnice neslúžia priamo na tvorbu a prácu s neurónovými sieťami.

***

# Úloha č.2:

### Znenie

Otestujte, či vaše zariadenie obsahuje CUDA-enabled GPU, ak áno použite ho. Dplňte nasledujúci fragment kódu.

### Riešenie

In [ ]:
if torch.cuda.is_available():
    devide = torch.device('cuda')
else:
    devide = torch.device('cpu')

### Vysvetlenie

Bol vytvorený objekt *troch.device*, slúžiaci na špecifikovanie zariadenia, na ktorom budú dáta/model siete ukladané. Objekt dostáva ako svoj vstupný parameter názov požadovaného zariadnia, za pomoci ternárneho operátora a funkcie *torch.cuda.is_available()*, ktorá vracia *True*, ak vaše zariadenie obsahuje CUDA-enabled GPU.

***

# Úloha č.3:

### Znenie

Získajte dataset **CIFAR10** ponúkaný balíkom torchvision. Preskúmajte jeho štruktúru a formu dát ktoré obsahuje.

Dokumentáciu k datasetu: https://pytorch.org/vision/stable/generated/torchvision.datasets.CIFAR10.html#torchvision.datasets.CIFAR10

### Riešenie

In [ ]:
batch_size = 100

# dataset with training data
train_dataset = torchvision.datasets.CIFAR10(
    root='./Data',
    download=True,
    train=True,
    transform=transforms.ToTensor()
)

# dataset with testing data
test_dataset = torchvision.datasets.CIFAR10(
    root='./Data',
    download=True,
    train=False,
    transform=transforms.ToTensor()
)

# dataloader for training data
train_loader = torch.utils.data.DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True
)

# dataloader for testing data
test_loader = torch.utils.data.DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print(train_dataset[0][0].shape)

### Vysvetlenie

Vytvorené boli dva *Dataset* objekty datasetu CIFAR10, jeden obsahujúci trénovacie a druhý testovacie dáta, podľa parametru konštruktora. Oba datasety vykonávajú transformáciu *ToTensor()*, ktorá obrázkové dáta transformuje na Pytorch tensory. Pre ne boli následne vytvorené dva *DataLoader* objekty. 

Dáta obsiahnuté v takomto datasete majú tvar (3, 32, 32), keďže ide o RGB obrázky o veľkosti 32\*32.

### *Doplňujúca úloha*

*Pomocou knižnice matplotlib sa pokúste vizualizovať dáta v datasete*

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 10))

for i in range(2):
    for j in range(2):
        img, label = test_dataset[2*i + j]
        axes[i, j].imshow(img[0], cmap="Greys_r")

***

# Úloha č.4:

### Znenie

Vytvorte triedu vašej siete. Nezabudnite v nej implementovať **konštruktor** a metódu **forward**, obdobne ako tomu bolo vo vzorovej úlohe. Experimentujte pri tom so štruktúrou siete (skrytých vrstiev).

**Nezabudnite správne určiť hyperparametre siete (na základe veľkosti vstupných obrázkov a počtu tried)**

### Riešenie

In [ ]:
input_size = 32 * 32 * 3
num_classes = 10

class MyNet(nn.Module):
    def __init__(self, input_size, num_classes):
        super(MyNet, self).__init__()

        self.fc_hidden1 = nn.Linear(input_size, 3000)
        self.fc_hidden2 = nn.Linear(3000, 2000)
        self.fc_hidden3 = nn.Linear(2000, 500)
        self.fc_output = nn.Linear(500, num_classes)     
    
    def forward(self, x):
        out = F.relu(self.fc_hidden1(x))
        out = F.relu(self.fc_hidden2(out))
        out = F.relu(self.fc_hidden3(out))
        out = self.fc_output(out)
        return out 
    
model = NeuralNet(input_size, num_classes).to(device)

### Vysvetlenie

Bola vytvorená sieť obsahujúca štyri lineárne vrstvy. Vstupná veľkosť vstupnej vrstvy je 3 * 32 * 32 (3 RGB kanály na obrázok, obrázky o veľkosti 32 * 32) a výstupna veľkosť je 10 (pretože CIFAR-10 obsahuje dáta delené do desiatich tried).

***
# Úloha č.5:

### Znenie

Keď ste si navrhli model vašej siete, doplňte nasledujúci kód a implementujte tak trénovaciu slučku siete. Vychádzajte zo štruktúry slučky ukázanej v kroku 1.5. Kód doplňte o hyperparamter, vytvorenie chybovej funkcie, optimizátora a krok vykonania predikcie, vypočítania chyby a vykonanie spätného kroku. 

### Riešenie

In [ ]:
model = MyNet(input_size, num_classes).to(device)

# training hyperparameters
num_epochs = 20
learning_rate = 0.001

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

start_time = time.time()
for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(tqdm(train_loader)): 
        images = images.reshape(-1, 32 * 32 * 3).to(device)
        labels = labels.to(device)
        
        outputs = model(images)

        optimizer.zero_grad()
        
        loss = criterion(outputs, labels)
        loss.backward()
        
        optimizer.step()
    
    if epoch + 1 == num_epochs: 
        print (f'\nTraining took {str(timedelta(seconds=time.time() - start_time))}, loss after all epochs: {loss.item():.5f}\n')    
    else:
        print (f'Loss at end of epoch {epoch+1}: {loss.item():.5f}\n')

### Vysvetlenie

Bola implementovaná tradičná trénovacia slučka, ktorá pre stanovený počet epoch iteratívne prechádza trénovacími dátami, vykonáva na nich predpovede a na základe nich pomocou optimalizačného algoritmu upravuje váhy v sieti. Konkrétne bol použitý optimizátor *Adam* a stratová funkcia *CrossEntropyLoss*.

*Pozn: dáta sú pred prechodom sieťou pretvarované na tvar (-1, 32 * 32 * 3). Takáto transformácia má za následok to, že dáta sú pretvarované na dvojrozmerný tensor, obsahujúci ľubovoľne veľa (-1 slúži ako placeholder), jednorozmerných tensorov, s dĺžkou podľa veľkosti obrázkov. Teda z dávky trojrozmerných tensorov vzniká dávka jednorozmerných, vyrovnaných tensorov.*

***
# Úloha č.6:

### Znenie

Po úspešnom natrénovaní siete implementuje trénovaciu slučku siete. Opäť vychádzajte zo slučky ukázanej v kroku 1.6, ktorá vypočítava percento správne označkovaných testovacích dát.

### Riešenie

In [ ]:
with torch.no_grad(): # don't adjust gradients (to reduce memory usage)   
    correct_count = 0
    total_count = 0

    # for each batch, make predictions and calculate how many of them were correct
    for images, labels in tqdm(test_loader):
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        _, predicted_labels = torch.max(outputs.data, 1)
        
        total_count += labels.size(0)
        correct_count += (predicted_labels == labels).sum()
        
accuracy = 100. * correct_count / total_count
print (f'Messured accuraccy: {accuracy:0.3f}% \n')

### Vysvetlenie

Bola vytvorená jednoduchá testovacia slučka, ktorá iteratívne vykonáva predickiu všetkých testovacích dát na natrénovanej sieti. Výslednou metrikou presnosti natrénovanej siete je pomer správne vykonaných a všetkých predikcií. Na iterovanie testovacími dátami bol použítý na to špecificky vytvorený *test_loader*, obsahujúci iba testovacie dáta.